In [ ]:
# System Dependencies
from __future__ import annotations
from pathlib import Path
import os, sys, io, json, string, time, pathlib, re, requests
import argparse, hashlib, base64, unicodedata, mimetypes
from dotenv import load_dotenv, find_dotenv

# Client and PDF packages
from supabase import create_client, Client
from PIL import Image
from IPython.display import IFrame, display, HTML, JSON
import pandas as pd
import pymupdf as fitz

# Chunking
from langchain_text_splitters import SpacyTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

def find_project_root() -> Path:
    p = Path.cwd()
    markers = {".git", "pyproject.toml", ".env"}
    for up in [p, *p.parents]:
        if any((up / m).exists() for m in markers):
            return up
    return p

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

env_path = find_dotenv(filename=".env", usecwd=True) or str(PROJECT_ROOT / ".env")
print("Loaded .env from:", env_path)
load_dotenv(env_path, override=False)

# Variables and helpers import
from ingest.constants import SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, STORAGE_BUCKET, PDF_FILENAME# , OVERLAP_CHARS, TARGET_CHARS
from ingest.chunk_import import slug, norm_title, extract_text_pages, windows, flatten, descendants

# Functions to test
from ingest.chunk_import import fetch_pdf_from_storage, build_toc, chapter_intervals, chunk_sections
from ingest.embedding_import import generate_embeddings, validate_embeddings

In [ ]:
'''Supabase PDF Retrieval'''

def fetch_pdf_from_storage(
    supabase_url: str,
    service_role_key: str,
    bucket: str,
    filename: str
) -> bytes:
    """
    Fetch PDF from Supabase Storage using service_role key.
    Works for private buckets.
    """
    auth_url = f"{supabase_url}/storage/v1/object/authenticated/{bucket}/{filename}"
    
    headers = {
        "apikey": service_role_key,
        "Authorization": f"Bearer {service_role_key}"
    }
    
    print(f"📥 Fetching PDF...")
    resp = requests.get(auth_url, headers=headers, timeout=30)
    resp.raise_for_status()
    
    pdf_bytes = resp.content
    if not pdf_bytes.startswith(b'%PDF'):
        raise ValueError("Downloaded file is not a valid PDF")
    
    print(f"✅ Downloaded {len(pdf_bytes):,} bytes ({len(pdf_bytes) / 1024 / 1024:.2f} MB)")
    return pdf_bytes

# Fetch PDF
pdf_bytes = fetch_pdf_from_storage(
    SUPABASE_URL,
    SUPABASE_SERVICE_ROLE_KEY,
    STORAGE_BUCKET,
    PDF_FILENAME
)

# Generate doc key
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]
print(f"Doc key: {doc_key}")

# Display PDF directly from bytes using data URL
print("\n📄 Displaying PDF:")
pdf_base64 = base64.b64encode(pdf_bytes).decode('utf-8')
display(HTML(f'<iframe src="data:application/pdf;base64,{pdf_base64}" width="800" height="600"></iframe>'))

In [ ]:
'''ToC Function'''

def build_toc(pdf_bytes: bytes) -> Tuple[str, List[dict], int]:
    """
    Build hierarchical ToC from PDF bytes.
    Returns: (doc_key, toc_tree, page_count)
    """
    doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]
    
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        toc_raw = doc.get_toc(simple=True) or []
        last_page = doc.page_count
        
        toc, stack = [], []
        idx = {1:0, 2:0, 3:0, 4:0, 5:0, 6:0}
        
        def push_node(level: int, title: str, page_start: int):
            for l in range(level, 7):
                idx[l] = idx[l] + 1 if l == level else 0
            path = "-".join(str(idx[l]) for l in range(1,7) if idx[l] > 0)
            node = {
                "id": f"h{level}-{path}__{slug(title)}",
                "title": title,
                "level": level,
                "page_start": page_start,
                "page_end": None,
                "children": []
            }
            while stack and stack[-1]["level"] >= level:
                stack[-1]["page_end"] = max(page_start - 1, stack[-1]["page_start"])
                stack.pop()
            (toc if not stack else stack[-1]["children"]).append(node)
            stack.append(node)
        
        for lvl, title, p1 in toc_raw:
            push_node(int(lvl), norm_title(title), int(p1))
        
        while stack:
            stack[-1]["page_end"] = last_page
            stack.pop()
    
    return doc_key, toc, last_page

doc_key, toc, page_count = build_toc(pdf_bytes)

# Print the TOC as formatted JSON
print(json.dumps(toc, indent=2))

# Or print all outputs
print(f"Doc Key: {doc_key}")
print(f"Page Count: {page_count}")
print(f"\nToC as JSON:")
print(json.dumps(toc, indent=2))

In [ ]:
'''Chunk Inspection'''
# The RecursiveCharacterTextSplitter attempts to keep larger units (e.g., paragraphs) intact. 
# If a unit exceeds the chunk size, it moves to the next level (e.g., sentences). This process continues down to the word level if necessary.


def chunk_sections(
    pdf_bytes: bytes,
    toc: List[dict],
    chunk_size: int = 1000,
    overlap: int = 150
) -> List[dict]:
    """
    Chunk all H1 sections and their children.
    Returns list of chunk dicts.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "]
    )
    
    chunks = []
    flat_nodes = flatten(toc)
    
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        seq_by_section = {}
        
        for chapter in [n for n in flat_nodes if n["level"] == 1]:
            work = chapter_intervals(chapter)
            
            for page_start, page_end, sec in work:
                if page_start > page_end:
                    continue
                
                text = extract_text_pages(doc, page_start, page_end)
                if not text:
                    continue
                
                # Window text to avoid SpaCy limits
                docs = []
                for slab in windows(text, size=200_000, overlap=1_000):
                    docs.extend(splitter.create_documents([slab]))
                
                sid = sec["id"]
                seq_by_section[sid] = seq_by_section.get(sid, 0)
                
                for d in docs:
                    chunk_text = (d.page_content or "").strip()
                    if len(chunk_text) < 100:
                        continue
                    
                    seq_by_section[sid] += 1
                    
                    chunks.append({
                        "section_id": sid,
                        "chunk_seq": seq_by_section[sid],
                        "section_title": sec["title"],
                        "level": sec["level"],
                        "page_start": page_start,
                        "page_end": page_end,
                        "text": chunk_text
                    })
    
    return chunks

chunks = chunk_sections(pdf_bytes, toc, chunk_size=1000, overlap=150)

print(f"✅ Generated {len(chunks)} chunks\n")

# Interactive collapsible JSON viewer
display(JSON(chunks))  # Show first 5 chunks

In [ ]:


def chunk_sections_spacy(
    pdf_bytes: bytes,
    toc: List[dict],
    chunk_size: int = 1000,
    chunk_overlap: int = 2  # Number of sentences to overlap
) -> List[dict]:
    """
    Chunk using Spacy sentence boundaries with smart overlap.
    """
    # Split on sentences using spaCy
    splitter = SpacyTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        pipeline="en_core_web_sm"  # Make sure to: python -m spacy download en_core_web_sm
    )
    
    chunks = []
    flat_nodes = flatten(toc)
    
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        seq_by_section = {}
        
        for chapter in [n for n in flat_nodes if n["level"] == 1]:
            work = chapter_intervals(chapter)
            
            for page_start, page_end, sec in work:
                if page_start > page_end:
                    continue
                
                text = extract_text_pages(doc, page_start, page_end)
                if not text:
                    continue
                
                # Split by sentences
                docs = splitter.create_documents([text])
                
                sid = sec["id"]
                seq_by_section[sid] = seq_by_section.get(sid, 0)
                
                for d in docs:
                    chunk_text = (d.page_content or "").strip()
                    if len(chunk_text) < 100:
                        continue
                    
                    seq_by_section[sid] += 1
                    
                    chunks.append({
                        "section_id": sid,
                        "chunk_seq": seq_by_section[sid],
                        "section_title": sec["title"],
                        "level": sec["level"],
                        "page_start": page_start,
                        "page_end": page_end,
                        "text": chunk_text
                    })
    
    return chunks

chunks = chunk_sections_spacy(pdf_bytes, toc)

print(f"✅ Generated {len(chunks)} chunks\n")

# Interactive collapsible JSON viewer
display(JSON(chunks))  # Show first 5 chunks

In [ ]:
def get_section_hierarchy(sec: dict, toc: List[dict]) -> List[dict]:
    """
    Get the full hierarchy path from root to this section.
    Returns list of nodes from highest level to current section.
    """
    hierarchy = []
    
    def find_parents(node: dict, all_nodes: List[dict]) -> List[dict]:
        """Recursively find all parent sections"""
        parents = []
        
        # Find direct parent (highest level node that contains this page range)
        for candidate in all_nodes:
            if (candidate["level"] < node["level"] and 
                candidate["page_start"] <= node["page_start"] and
                candidate["page_end"] >= node["page_end"]):
                # This is a parent, now find ITS parents
                parents.extend(find_parents(candidate, all_nodes))
                parents.append(candidate)
                break
        
        return parents
    
    flat_nodes = flatten(toc)
    hierarchy = find_parents(sec, flat_nodes)
    hierarchy.append(sec)  # Add the section itself
    
    return hierarchy


def build_markdown_headers(hierarchy: List[dict]) -> str:
    """Convert hierarchy to markdown headers"""
    headers = []
    for node in hierarchy:
        prefix = "#" * node["level"]
        headers.append(f"{prefix} {node['title']}")
    return "\n".join(headers)


def chunk_by_toc_structure_v2(
    pdf_bytes: bytes,
    toc: List[dict],
    max_chunk_size: int = 2000
) -> List[dict]:
    """
    Use ToC structure with proper markdown headers injected.
    """
    from langchain.text_splitter import SpacyTextSplitter
    
    chunks = []
    flat_nodes = flatten(toc)
    
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        for sec in flat_nodes:
            text = extract_text_pages(doc, sec["page_start"], sec["page_end"])
            if not text:
                continue
            
            # Get full hierarchy
            hierarchy = get_section_hierarchy(sec, toc)
            markdown_headers = build_markdown_headers(hierarchy)
            
            # Combine headers + content
            full_text = f"{markdown_headers}\n\n{text.strip()}"
            
            # Chunk if needed
            if len(full_text) <= max_chunk_size:
                chunks.append({
                    "section_id": sec["id"],
                    "chunk_seq": 1,
                    "section_title": sec["title"],
                    "level": sec["level"],
                    "page_start": sec["page_start"],
                    "page_end": sec["page_end"],
                    "hierarchy": [n["title"] for n in hierarchy],
                    "text": full_text
                })
            else:
                splitter = SpacyTextSplitter(chunk_size=max_chunk_size - len(markdown_headers))
                body_docs = splitter.create_documents([text])
                
                for i, d in enumerate(body_docs, 1):
                    chunk_text = f"{markdown_headers}\n\n{d.page_content.strip()}"
                    
                    chunks.append({
                        "section_id": sec["id"],
                        "chunk_seq": i,
                        "section_title": sec["title"],
                        "level": sec["level"],
                        "page_start": sec["page_start"],
                        "page_end": sec["page_end"],
                        "hierarchy": [n["title"] for n in hierarchy],
                        "text": chunk_text
                    })
    
    return chunks

chunks = chunk_by_toc_structure_v2(pdf_bytes, toc)

print(f"✅ Generated {len(chunks)} chunks\n")

# Interactive collapsible JSON viewer
display(JSON(chunks))  # Show first 5 chunks

In [ ]:
def get_ancestor_titles(node: dict, flat_nodes: List[dict]) -> List[Tuple[int, str]]:
    """Get all ancestor section titles with their levels."""
    ancestors = []
    current_level = node["level"]
    current_page = node["page_start"]
    
    # Find all ancestors by walking backwards through nodes
    for n in reversed(flat_nodes):
        if n["page_start"] <= current_page and n["level"] < current_level:
            ancestors.insert(0, (n["level"], n["title"]))
            current_level = n["level"]
            if current_level == 1:
                break
    
    return ancestors

def chunk_by_toc_with_context(
    pdf_bytes: bytes,
    toc: List[dict],
    chunk_size: int = 2000,
    chunk_overlap: int = 50
) -> List[dict]:
    """
    Use ToC structure with full hierarchical headers + SpaCy sentence splitting.
    """
    # Use SpaCy for sentence-aware splitting
    splitter = SpacyTextSplitter(
        pipeline="en_core_web_sm",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    chunks = []
    flat_nodes = flatten(toc)
    
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        seq_by_section = {}
        
        for chapter in [n for n in flat_nodes if n["level"] == 1]:
            work = chapter_intervals(chapter)
            
            for page_start, page_end, sec in work:
                if page_start > page_end:
                    continue
                
                text = extract_text_pages(doc, page_start, page_end)
                if not text:
                    continue
                
                # Build full header hierarchy
                ancestors = get_ancestor_titles(sec, flat_nodes)
                header_parts = []
                
                # Add ancestor headers
                for level, title in ancestors:
                    header_parts.append(f"{'#' * level} {title}")
                
                # Add current section header
                header_parts.append(f"{'#' * sec['level']} {sec['title']}")
                
                header = "\n\n".join(header_parts) + "\n\n"
                
                # Window text to avoid SpaCy limits (E088 error)
                docs = []
                for slab in windows(text, size=200_000, overlap=1_000):
                    # Combine header with content for first window only
                    if not docs:
                        windowed_text = header + slab
                    else:
                        windowed_text = slab
                    
                    docs.extend(splitter.create_documents([windowed_text]))
                
                sid = sec["id"]
                seq_by_section[sid] = seq_by_section.get(sid, 0)
                
                for i, d in enumerate(docs):
                    chunk_text = (d.page_content or "").strip()
                    if len(chunk_text) < 100:
                        continue
                    
                    seq_by_section[sid] += 1
                    
                    chunks.append({
                        "section_id": sid,
                        "chunk_seq": seq_by_section[sid],
                        "section_title": sec["title"],
                        "level": sec["level"],
                        "page_start": page_start,
                        "page_end": page_end,
                        "text": chunk_text,
                        "hierarchy": [t for _, t in ancestors] + [sec["title"]]
                    })
    
    return chunks

chunks = chunk_by_toc_with_context(pdf_bytes, toc)
print(f"✅ Generated {len(chunks)} chunks\n")
# Interactive collapsible JSON viewer
display(JSON(chunks))  # Show first 5 chunks

In [ ]:
df = pd.DataFrame(chunks)

# Add computed metrics
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['sentence_count'] = df['text'].str.count(r'[.!?]+')
df['page_span'] = df['page_end'] - df['page_start'] + 1

print("\n🔍 CONTENT QUALITY CHECKS")
print("="*80)

# Check for very short chunks (likely errors)
short_chunks = df[df['text_length'] < 100]
print(f"\n⚠️  Chunks under 100 chars: {len(short_chunks)}")
if len(short_chunks) > 0:
    print("\nSample short chunks:")
    for idx, row in short_chunks.head(3).iterrows():
        print(f"  - {row['section_title']}: {row['text'][:80]}...")

# Check for very long chunks (might need splitting)
long_chunks = df[df['text_length'] > 2000]
print(f"\n⚠️  Chunks over 2000 chars: {len(long_chunks)}")
if len(long_chunks) > 0:
    print("\nLongest chunks:")
    print(long_chunks[['section_title', 'text_length']].head())

# Check for incomplete sentences (chunks ending mid-sentence)
incomplete = df[~df['text'].str.endswith(('.', '!', '?', '"', "'", ')', ']'))]
print(f"\n⚠️  Chunks with incomplete endings: {len(incomplete)}")
print(f"   Percentage: {len(incomplete)/len(df)*100:.1f}%")

# Check for chunks that might start mid-sentence
mid_sentence_start = df[df['text'].str.match(r'^[a-z]')]
print(f"\n⚠️  Chunks starting with lowercase: {len(mid_sentence_start)}")
print(f"   Percentage: {len(mid_sentence_start)/len(df)*100:.1f}%")